# Session 4 — Lists, Dictionaries and Tuples

**Goal of this session:** hold a real dataset in memory without inventing a hundred variable names.

*Python for Neuroscience, session 4 of 12.*

## Why this matters

An electrophysiology recording is not one number. It is a set of spike times per neuron, plus the region each neuron came from, plus the subject, plus the condition on each trial.

Python gives you a few containers, and picking the right one is most of the battle. By the end of this session you will build a small dataset of several neurons and plot it as a raster, which is the standard figure in spiking work.

## Lists

A list is an ordered box of things. Square brackets, commas between items. Here are the times, in seconds, when one neuron fired.

In [ ]:
spike_times = [0.12, 0.19, 0.33, 0.51, 0.52, 0.78, 1.04, 1.31, 1.33, 1.90]

print(spike_times)
print(len(spike_times), "spikes")

You get items out by position, counting from zero. Negative numbers count from the end.

In [ ]:
print(spike_times[0])     # first spike
print(spike_times[3])     # fourth spike
print(spike_times[-1])    # last spike

A colon takes a slice. The rule to memorise: the start is included, the end is not.

In [ ]:
print(spike_times[0:3])   # first three
print(spike_times[:3])    # same thing
print(spike_times[5:])    # from the sixth onward

Lists can be changed after you make them, which is why the append pattern from last session works.

In [ ]:
spike_times.append(2.15)
print(spike_times[-3:])

## Dictionaries

A list is fine for one neuron's spike times. It is terrible for describing the neuron itself, because you would have to remember that position 2 means the brain region.

A dictionary stores values under names. Curly braces, and `key: value` pairs.

In [ ]:
neuron = {
    "id": "V1_042",
    "region": "V1",
    "depth_um": 620,
    "spike_times": [0.12, 0.19, 0.33, 0.51, 0.78],
}

print(neuron["region"])
print(neuron["spike_times"])
print(len(neuron["spike_times"]), "spikes")

In [ ]:
# add a field, then look at what fields exist
neuron["quality"] = "good"
print(list(neuron.keys()))

## Tuples

A tuple is a list that cannot be changed. Round brackets instead of square ones. Use it for things that genuinely should not move, like a coordinate or a fixed time window.

In [ ]:
window = (0.0, 2.0)          # analysis window, start and end
coords = (-42, -22, 58)      # MNI coordinates of a region

start, end = window          # unpacking, very common in Python
print(f"Analysing from {start} to {end} seconds")

## Nesting them into a dataset

Here is the useful part. A list of dictionaries is a small dataset: each dictionary is one neuron, and the list holds all of them.

This is exactly the shape that real recording software exports.

In [ ]:
dataset = [
    {"id": "V1_042", "region": "V1",
     "spike_times": [0.12, 0.19, 0.33, 0.51, 0.78, 1.04, 1.31, 1.90]},
    {"id": "V1_043", "region": "V1",
     "spike_times": [0.05, 0.41, 0.44, 0.47, 0.95, 1.55, 1.58, 1.61, 1.99]},
    {"id": "HPC_007", "region": "Hippocampus",
     "spike_times": [0.31, 0.88, 1.42, 1.77]},
    {"id": "M1_019", "region": "Motor",
     "spike_times": [0.02, 0.09, 0.15, 0.22, 0.60, 0.66, 0.72, 1.10, 1.18,
                     1.25, 1.68, 1.74, 1.81]},
]

print(len(dataset), "neurons")
print(dataset[2]["id"], "is in", dataset[2]["region"])

Now a loop over the dataset gives you a summary table in four lines.

In [ ]:
recording_duration = 2.0

for neuron in dataset:
    n = len(neuron["spike_times"])
    rate = n / recording_duration
    print(f"{neuron['id']:10s} {neuron['region']:12s} {n:3d} spikes  {rate:5.1f} Hz")

## The raster plot

A raster puts one row per neuron and one vertical tick per spike. It is the first plot anyone makes with spiking data, and you now have everything you need to build it straight from your list of dictionaries.

`ax.eventplot()` does the drawing.

In [ ]:
import matplotlib.pyplot as plt

all_spikes = [neuron["spike_times"] for neuron in dataset]
labels = [neuron["id"] for neuron in dataset]

fig, ax = plt.subplots(figsize=(10, 4))
ax.eventplot(all_spikes, colors="#2b6cb0", linelengths=0.7, linewidths=2)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=12)
ax.set_xlabel("time (s)", fontsize=13)
ax.set_title("Spike raster, four neurons", fontsize=15)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

Look at M1_019 in the bottom row. The spikes come in clumps rather than spread out evenly. That is bursting, and you can see it in the figure without computing anything.

## Try it yourself

Add a fifth neuron to `dataset` with spike times of your own and rerun the plot. Nothing else needs to change, which is the whole point of holding data in a container instead of in loose variables.

**Next session:** NumPy, where these lists become arrays and everything gets faster.